In [1]:
#packages
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.dataframe
import pyspark.sql.functions as f
from pyspark.sql.functions import col
import math as m



In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/09 12:44:42 WARN Utils: Your hostname, Lachys-Laptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/10/09 12:44:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/09 12:44:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("Project 1")
    
    # === MEMORY MANAGEMENT ===
    .config("spark.driver.memory", "4g")          # Increase driver memory (safe for 16GB+ systems)
    .config("spark.executor.memory", "4g")        # Executors share same JVM locally
    .config("spark.driver.maxResultSize", "2g")   # Prevent large collect() results crashing driver

    # === PARALLELISM & SHUFFLING ===
    .config("spark.sql.shuffle.partitions", "48")  # Default is 200 — too high locally
    .config("spark.default.parallelism", "8")      # ~ number of cores on your system
    .config("spark.sql.files.maxPartitionBytes", "128MB")  # Optimal shuffle partition size

    # === PERFORMANCE TUNING ===
    .config("spark.memory.fraction", "0.85")        # 85% of JVM heap for Spark execution
    .config("spark.memory.storageFraction", "0.3")  # 30% of execution memory for caching
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # Fast pandas conversion

    # === TEMP STORAGE ===
    .config("spark.local.dir", "/tmp/spark-temp")   # Disk spill location for large shuffles

    # === DEFAULT OPTIONS ===
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", True)
    .config("spark.sql.session.timeZone", "Etc/UTC")

    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/09 12:24:37 WARN Utils: Your hostname, Lachys-Laptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/10/09 12:24:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/09 12:24:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/09 12:24:38 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


In [3]:
#reading in data

tbl_merchants_raw = spark.read.parquet('../data/tables/merchant_data/tbl_merchants.parquet')
consumer_user_details = spark.read.parquet('../data/tables/merchant_data/consumer_user_details.parquet')
transactions21 = spark.read.parquet('../data/tables/transaction_data/transactions_20210228_20210827_snapshot/')
transactions2122 = spark.read.parquet('../data/tables/transaction_data/transactions_20210828_20220227_snapshot/')
transactions22 = spark.read.parquet('../data/tables/transaction_data/transactions_20220228_20220828_snapshot/')
con_fraud_prob = spark.read.option("header","true").csv('../data/tables/merchant_data/consumer_fraud_probability.csv')
merch_fraud_prob = spark.read.option("header", "true").csv('../data/tables/merchant_data/merchant_fraud_probability.csv')

tbl_consumer_raw = spark.read.option("header", "true").csv('../data/tables/merchant_data/tbl_consumer.csv')

transactions = transactions21.unionByName(transactions2122)
transactions = transactions.unionByName(transactions22)

In [4]:
# Load first CSV
postcodes_df = spark.read.csv("../data/income/2024 Locality to 2021 SA2 Coding Index.csv", header=True, inferSchema=True)

# Load second CSV
income_df = spark.read.csv("../data/income/sa2_income.csv", header=True, inferSchema=True)

In [5]:
#Functions
def find_NULL(dfs):

    """Finds any rows with NULLs over different datasets"""

    for df in dfs:
        condition = f.lit(False)
        for col_name in df.columns:
            condition = condition | f.col(col_name).isNull()

        df.filter(condition).show()
    return df.filter(condition).count()

def filter_outliers(data, variables):
    
    """filters outliers of continuous data"""

    n=data.count()
    for feature in variables:
        # Calculate Q1 and Q3
        quantiles = data.approxQuantile(feature, [0.25, 0.75], 0.01)
        q1, q3 = quantiles
        iqr = q3 - q1

        #from ADS lecture slides, n>>100
        scale = m.sqrt(m.log(n)) - 0.5
        if scale<3:
            scale=3
        lower_bound = q1 - scale * iqr
        upper_bound = q3 + scale * iqr
        if lower_bound<0:
            data = data.filter((col(feature) >= 0) & (col(feature) <= upper_bound))
        else:
            data = data.filter((col(feature) >= lower_bound) & (col(feature) <= upper_bound))
    
    return data

def spark_shape(self):
    
    """Easy function for shape of a spark df"""

    return (self.count(), len(self.columns))

pyspark.sql.dataframe.DataFrame.shape = property(spark_shape)

In [6]:
#cleaning tags
string = "name|address|state|postcode|gender|consumer_id"

# Clean consumer table
tbl_consumer = (
    tbl_consumer_raw
    .withColumn("cust_name", f.split(col(string), "\\|").getItem(0))
    .withColumn("address", f.split(col(string), "\\|").getItem(1))
    .withColumn("state", f.split(col(string), "\\|").getItem(2))
    .withColumn("postcode", f.split(col(string), "\\|").getItem(3))
    .withColumn("gender", f.split(col(string), "\\|").getItem(4))
    .withColumn("consumer_id", f.split(col(string), "\\|").getItem(5))
    .drop(string)
)


# Clean merchants table
tbl_merchants = (
    tbl_merchants_raw
    # remove leading (( or [[ and trailing )) or ]]
    .withColumn(
        "tags_clean",
        f.regexp_replace(
            "tags",
            r"^\(\(|^\(\[|^\[\(|^\[\[|\)\)$|\]\)$|\)\]$|\]\]$",
            ""
        )
    )
    # split on `), (` or `], [`
    .withColumn("tags_array", f.split("tags_clean", r"\)\s*,\s*\(|\]\s*,\s*\["))
    # extract each element
    .withColumn("biz_tags", f.lower(f.col("tags_array")[0]))
    .withColumn("rev_band", f.col("tags_array")[1])
    .withColumn("take_rate", f.regexp_extract(f.col("tags_array")[2], r"take rate:\s*([0-9.]+)", 1)
    )
    .drop("tags", "tags_clean", "tags_array")
)

tbl_merchants=tbl_merchants.withColumn("biz_tags", f.regexp_replace("biz_tags", "  ", " "))

In [7]:
#joining transactions and merchants datasets
merchant_transactions=transactions.join(tbl_merchants, on='merchant_abn', how='left')
print(merchant_transactions.shape)
find_NULL([merchant_transactions])

(14195505, 9)


+------------+-------+------------------+--------------------+--------------+----+--------+--------+---------+
|merchant_abn|user_id|      dollar_value|            order_id|order_datetime|name|biz_tags|rev_band|take_rate|
+------------+-------+------------------+--------------------+--------------+----+--------+--------+---------+
| 29566626791|      8| 74.15732460440282|71a81652-cc91-4bf...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 32234779638|  18490|107.14809429376949|20149572-a55b-41f...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 67202032418|     20| 55.46394975814555|a29071b4-29b3-4f2...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 32461318592|     23| 613.9306657410166|4b2e2154-65d8-44f...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 32234779638|     25| 87.15685629102919|e6763664-e95f-4eb...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 23633724513|     26|3459.2423030023524|fee9ead7-9ce2-4a4...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
|

580830

In [8]:
merchant_transactions = merchant_transactions.dropna()
print(merchant_transactions.shape)

(13614675, 9)


In [9]:
merchant_transactions.groupBy('name').count().orderBy("count", ascending=True).show()

+--------------------+-----+
|                name|count|
+--------------------+-----+
|Aliquam Eu Institute|    1|
|Lobortis Nisi Ass...|    1|
|Elit Dictum Eu Fo...|    1|
|Consequat Foundation|    1|
|Aenean Gravida In...|    1|
|       Phasellus LLP|    1|
|    Curae Foundation|    1|
|Integer Urna Inst...|    2|
|    Gravida Nunc LLP|    2|
|  Cras Convallis Ltd|    2|
|        Elit Limited|    2|
|            Elit LLP|    2|
|     Sem Corporation|    2|
|Semper Pretium Li...|    2|
|Consectetuer Indu...|    2|
|Adipiscing Fringi...|    2|
|    Massa Rutrum LLP|    2|
|Egestas Nunc Sed LLC|    2|
|Accumsan Laoreet ...|    2|
|Dictum Mi Corpora...|    2|
+--------------------+-----+
only showing top 20 rows


In [10]:
#filter outliers by biz_tag

n = merchant_transactions.count()
# compute the scale factor
scale = m.sqrt(m.log(n)) - 0.5
stats_by_band = (merchant_transactions.groupby('biz_tags')
                                      .agg(f.expr("percentile_approx(dollar_value, 0.25)").alias("Q1"),
                                           f.expr("percentile_approx(dollar_value, 0.75)").alias("Q3")
                ).withColumn("IQR", f.col("Q3") - f.col("Q1"))
                 .withColumn("lower_bound", f.col("Q1") - scale * f.col("IQR"))
                 .withColumn("upper_bound", f.col("Q3") + scale * f.col("IQR"))
                )
stats_by_band = stats_by_band.drop('IQR')

In [11]:
merchant_transactions = (
    merchant_transactions
    .join(stats_by_band, on="biz_tags", how="left")
    .filter(
        (col("dollar_value") >= col('lower_bound')) &
        (col("dollar_value") <= col("upper_bound"))
    )
    .select(merchant_transactions["*"])
)

merchant_transactions

merchant_abn,user_id,dollar_value,order_id,order_datetime,name,biz_tags,rev_band,take_rate
15549624934,2,130.3505283105634,6a84c3cf-612a-457...,2021-08-20,Commodo Associates,"opticians, optica...",c,2.76
46804135891,18482,6.6168976971833615,05b5edb5-b925-414...,2021-08-20,Suspendisse Dui C...,"opticians, optica...",c,2.93
11237511112,15,86.43306925925785,7b259290-89ef-441...,2021-08-20,Magna Institute,"opticians, optica...",c,2.11
48534649627,37,115.71011998439714,f850e292-c70d-431...,2021-08-20,Dignissim Maecena...,"opticians, optica...",a,6.64
22059270846,18563,13.552320313774313,6b3e02e0-baa0-4f3...,2021-08-20,Montes Nascetur R...,"opticians, optica...",a,6.59
46804135891,83,40.88736687931136,0f5df50c-5f44-46c...,2021-08-20,Suspendisse Dui C...,"opticians, optica...",c,2.93
81410315303,90,97.01139408070111,3f29db07-4c8c-402...,2021-08-20,Sed Dictum PC,"opticians, optica...",a,6.35
46804135891,96,71.58993600692456,bc0e620e-5b7b-42f...,2021-08-20,Suspendisse Dui C...,"opticians, optica...",c,2.93
60602272553,112,17.429130408212,b7bf5aaf-821d-411...,2021-08-20,Sagittis Duis Gra...,"opticians, optica...",b,4.93
46804135891,118,14.417591756971596,ef10a334-608d-491...,2021-08-20,Suspendisse Dui C...,"opticians, optica...",c,2.93


In [12]:
print(merchant_transactions.shape)

(13293844, 9)


In [13]:
merchant_transactions=merchant_transactions.withColumnRenamed('name', 'business')
merchant_transactions=merchant_transactions.drop('order_id')
merchant_transactions

merchant_abn,user_id,dollar_value,order_datetime,business,biz_tags,rev_band,take_rate
15549624934,2,130.3505283105634,2021-08-20,Commodo Associates,"opticians, optica...",c,2.76
46804135891,18482,6.6168976971833615,2021-08-20,Suspendisse Dui C...,"opticians, optica...",c,2.93
11237511112,15,86.43306925925785,2021-08-20,Magna Institute,"opticians, optica...",c,2.11
48534649627,37,115.71011998439714,2021-08-20,Dignissim Maecena...,"opticians, optica...",a,6.64
22059270846,18563,13.552320313774313,2021-08-20,Montes Nascetur R...,"opticians, optica...",a,6.59
46804135891,83,40.88736687931136,2021-08-20,Suspendisse Dui C...,"opticians, optica...",c,2.93
81410315303,90,97.01139408070111,2021-08-20,Sed Dictum PC,"opticians, optica...",a,6.35
46804135891,96,71.58993600692456,2021-08-20,Suspendisse Dui C...,"opticians, optica...",c,2.93
60602272553,112,17.429130408212,2021-08-20,Sagittis Duis Gra...,"opticians, optica...",b,4.93
46804135891,118,14.417591756971596,2021-08-20,Suspendisse Dui C...,"opticians, optica...",c,2.93


In [14]:
merchant_transactions.groupBy("biz_tags").agg(
    f.min("dollar_value").alias("min_value"),
    f.max("dollar_value").alias("max_value"),
    f.mean("dollar_value").alias("mean"),
    (f.max("dollar_value") - f.min("dollar_value")).alias("range")
).orderBy("mean", ascending=False).show()

+--------------------+--------------------+------------------+------------------+------------------+
|            biz_tags|           min_value|         max_value|              mean|             range|
+--------------------+--------------------+------------------+------------------+------------------+
|jewelry, watch, c...|   3.409793978681009| 46001.13901942742|  9278.56318565421| 45997.72922544874|
|art dealers and g...|  0.4127496907944707| 10335.94618503865| 1966.235783927586|10335.533435347856|
|             telecom|  0.2931526313090789| 11606.18761084434|1735.6703991984127| 11605.89445821303|
|equipment, tool, ...|0.040595292090802974| 8813.127778854296| 1261.652706482709| 8813.087183562206|
|stationery, offic...|0.004010169952587961| 2333.490127589658|456.97980173990925| 2333.486117419706|
|health and beauty...| 6.59812930332817E-4|1672.2945523725923| 294.9553375878973| 1672.293892559662|
|motor vehicle sup...|7.092782606876731E-4|1356.3937038188778| 271.4403187831411| 1356.3929

In [15]:
biz_tags_list = merchant_transactions.select("biz_tags").distinct().rdd.flatMap(lambda x: x).collect()
print(biz_tags_list)
print(len(biz_tags_list))

['opticians, optical goods, and eyeglasses', 'computer programming , data processing, and integrated systems design services', 'watch, clock, and jewelry repair shops', 'books, periodicals, and newspapers', 'digital goods: books, movies, music', 'antique shops - sales, repairs, and restoration services', 'art dealers and galleries', 'florists supplies, nursery stock, and flowers', 'equipment, tool, furniture, and appliance rent al and leasing', 'gift, card, novelty, and souvenir shops', 'cable, satellite, and other pay television and radio services', 'tent and awning shops', 'artist supply and craft shops', 'furniture, home furnishings and equipment shops, and manufacturers, except appliances', 'jewelry, watch, clock, and silverware shops', 'stationery, office supplies and printing and writing paper', 'telecom', 'computers, computer peripheral equipment, and software', 'hobby, toy and game shops', 'shoe shops', 'health and beauty spas', 'lawn and garden supply outlets, including nurser

In [ ]:
# Count how many distinct biz_tags each business has
biz_tag_counts = (
    merchant_transactions
    .groupBy("business")
    .agg(f.countDistinct("biz_tags").alias("distinct_tag_count"))
    .orderBy(f.desc("distinct_tag_count"))
)

# Show businesses (with more than one unique tag)
biz_tag_counts

[350.715s][warning][gc,alloc] Executor task launch worker for task 6.0 in stage 88.0 (TID 1227): Retried waiting for GCLocker too often allocating 35637 words


[383.150s][warning][gc,alloc] Executor task launch worker for task 8.0 in stage 98.0 (TID 1293): Retried waiting for GCLocker too often allocating 4194306 words


25/10/09 12:50:48 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.


business,distinct_tag_count
Donec Luctus Indu...,1
At Augue Corporation,1
Enim Etiam Indust...,1
Suspendisse Sed I...,1
Enim Non LLP,1
A Associates,1
Nisi A Odio Assoc...,1
Nulla Integer Vul...,1
Pede Malesuada Co...,1
Suspendisse Dui I...,1


In [ ]:
# Assigning the biz_tags to segments
merchant_transactions = merchant_transactions.withColumn(
    "segment",
    f.when(f.col("biz_tags").isin(
        "watch, clock, and jewelry repair shops",
        "jewelry, watch, clock, and silverware shops",
        "shoe shops",
        "antique shops - sales, repairs, and restoration services",
        "gift, card, novelty, and souvenir shops"
    ), "Fashion, Jewelry & Personal Goods")
   
    .when(f.col("biz_tags").isin(
        "books, periodicals, and newspapers",
        "digital goods: books, movies, music",
        "music shops - musical instruments, pianos, and sheet music",
        "art dealers and galleries",
        "artist supply and craft shops",
        "hobby, toy and game shops",
        "cable, satellite, and other pay television and radio services"
    ), "Arts, Media & Entertainment")
   
    .when(f.col("biz_tags").isin(
        "computers, computer peripheral equipment, and software",
        "computer programming , data processing, and integrated systems design services",
        "telecom",
        "equipment, tool, furniture, and appliance rent al and leasing",
        "stationery, office supplies and printing and writing paper"
    ), "Technology & Professional Services")
   
    .when(f.col("biz_tags").isin(
        "furniture, home furnishings and equipment shops, and manufacturers, except appliances",
        "tent and awning shops",
        "lawn and garden supply outlets, including nurseries",
        "florists supplies, nursery stock, and flowers"
    ), "Home, Garden & Living")
   
    .when(f.col("biz_tags").isin(
        "opticians, optical goods, and eyeglasses",
        "health and beauty spas",
        "bicycle shops - sales and service",
        "motor vehicle supplies and new parts"
    ), "Lifestyle, Health & Recreation")
   
    .otherwise("Other")
)
merchant_transactions

In [ ]:
merchant_transactions.write.parquet("../data/curated/merchant_transactions", mode="overwrite")

In [19]:
tbl_consumer=tbl_consumer.drop('address', 'cust_name', 'gender')
#tbl_consumer

In [ ]:
mtc_fraud1=merchant_transactions.join(merch_fraud_prob, on=['merchant_abn', 'order_datetime'], how='left')
mtc_fraud1=mtc_fraud1.withColumnRenamed('fraud_probability', 'merch_fraud_prob')
mtc_fraud=mtc_fraud1.join(con_fraud_prob, on=['user_id', 'order_datetime'], how='left')
mtc_fraud=mtc_fraud.withColumnRenamed('fraud_probability', 'con_fraud_prob')
mtc_fraud

In [21]:
mtc_fraud

user_id,order_datetime,merchant_abn,dollar_value,business,biz_tags,rev_band,take_rate,segment,merch_fraud_prob,con_fraud_prob
2,2021-08-20,15549624934,130.3505283105634,Commodo Associates,"opticians, optica...",c,2.76,"Lifestyle, Health...",NULL,NULL
18482,2021-08-20,46804135891,6.6168976971833615,Suspendisse Dui C...,"opticians, optica...",c,2.93,"Lifestyle, Health...",NULL,NULL
15,2021-08-20,11237511112,86.43306925925785,Magna Institute,"opticians, optica...",c,2.11,"Lifestyle, Health...",NULL,NULL
37,2021-08-20,48534649627,115.71011998439714,Dignissim Maecena...,"opticians, optica...",a,6.64,"Lifestyle, Health...",NULL,NULL
18563,2021-08-20,22059270846,13.552320313774313,Montes Nascetur R...,"opticians, optica...",a,6.59,"Lifestyle, Health...",NULL,NULL
83,2021-08-20,46804135891,40.88736687931136,Suspendisse Dui C...,"opticians, optica...",c,2.93,"Lifestyle, Health...",NULL,NULL
90,2021-08-20,81410315303,97.01139408070111,Sed Dictum PC,"opticians, optica...",a,6.35,"Lifestyle, Health...",NULL,NULL
96,2021-08-20,46804135891,71.58993600692456,Suspendisse Dui C...,"opticians, optica...",c,2.93,"Lifestyle, Health...",NULL,NULL
112,2021-08-20,60602272553,17.429130408212,Sagittis Duis Gra...,"opticians, optica...",b,4.93,"Lifestyle, Health...",NULL,NULL
118,2021-08-20,46804135891,14.417591756971596,Suspendisse Dui C...,"opticians, optica...",c,2.93,"Lifestyle, Health...",NULL,NULL


In [22]:
curated=mtc_fraud.groupBy(['merchant_abn', 'user_id']).agg(
    f.count('*').alias('count'),
    f.mean('dollar_value').alias('mean'),
    f.mean('merch_fraud_prob').alias('merch_fraud_prob'),
    f.mean('con_fraud_prob').alias('con_fraud_prob')
)
curated

[834.705s][warning][gc,alloc] Executor task launch worker for task 2.0 in stage 150.0 (TID 1639): Retried waiting for GCLocker too often allocating 24353 words


merchant_abn,user_id,count,mean,merch_fraud_prob,con_fraud_prob
22059270846,13210,1,73.45764111772307,NULL,NULL
11566786699,13328,3,40.22362138769044,NULL,NULL
46804135891,13810,10,35.42333163566343,NULL,NULL
98472033309,23273,1,141.6695357802577,NULL,NULL
41251795489,5302,1,4.254438362964493,NULL,NULL
46804135891,5629,11,28.608140325003465,NULL,NULL
53123395285,16783,2,39.8236657659222,NULL,NULL
46804135891,7861,6,19.520899453467013,NULL,NULL
48534649627,17642,2,55.84436760035899,NULL,NULL
46804135891,9914,12,23.88372594786503,NULL,NULL


In [23]:
new_curated=curated.join(consumer_user_details, on='user_id', how='left')
new_curated=new_curated.join(tbl_consumer, on='consumer_id', how='left')
new_curated=new_curated.drop('consumer_id')
new_curated

user_id,merchant_abn,count,mean,merch_fraud_prob,con_fraud_prob,state,postcode
667,10023283211,1,102.62436872103531,NULL,NULL,QLD,4705
895,10023283211,1,149.56486309776136,NULL,NULL,SA,5572
101,10023283211,1,483.59840138297426,NULL,NULL,QLD,4753
681,10023283211,1,423.2649125380065,NULL,NULL,WA,6956
146,10023283211,1,431.8016693626082,NULL,NULL,QLD,4467
259,10023283211,1,428.13919765470234,NULL,NULL,VIC,3559
571,10023283211,1,290.31547507345846,NULL,NULL,VIC,3225
1003,10023283211,1,22.108292788840235,NULL,NULL,VIC,3636
994,10023283211,1,364.4744699896163,NULL,NULL,VIC,3380
1664,10023283211,1,324.2484192944088,NULL,NULL,VIC,3063


In [24]:
print(new_curated.shape)

(7881927, 8)


In [ ]:
new_curated.write.parquet("data/curated/agg_by_userbiz", mode="overwrite")

In [ ]:
# Rename columns in df2 for easier handling
income_clean = (
    income_df
    .withColumnRenamed("Statistical Areas Level 2 2021 code", "SA2_CODE_2021")
    .withColumnRenamed("Statistical Areas Level 2 2021 name", "SA2_NAME_2021")
)

# Make sure join keys are the same type
postcodes_df = postcodes_df.withColumn("SA2_CODE_2021", col("SA2_CODE_2021").cast("string"))

income_clean = income_clean.withColumn("SA2_CODE_2021", col("SA2_CODE_2021").cast("string"))

income_clean = income_clean.filter(col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`") != 0)

# Perform join on SA2 code
merged_df = postcodes_df.join(income_clean, on="SA2_CODE_2021", how="left")
merged_df = merged_df.drop(income_clean.SA2_NAME_2021)  # drop df2’s version

missing_count = merged_df.filter(col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`").isNull()).count()
print(missing_count)

# Dropping NULL values in income column
print(merged_df.count()) # count before dropping NULL values
merged_df = merged_df.filter(col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`").isNotNull() )
print(merged_df.count()) # count after dropping NULL values

result_df = (
    merged_df
    .groupBy(col("POSTCODE").alias("postcode"))
    .agg(
        f.avg(
            col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`")
        ).alias("median_total_income_2020")
    )
)

In [ ]:
result_df = result_df.withColumn(
    "income_bin",
    f.when(f.col("median_total_income_2020") < 30000, "<30k")
     .when((f.col("median_total_income_2020") >= 30000) & (f.col("median_total_income_2020") < 40000), "30-40k")
     .when((f.col("median_total_income_2020") >= 40000) & (f.col("median_total_income_2020") < 50000), "40-50k")
     .when((f.col("median_total_income_2020") >= 50000) & (f.col("median_total_income_2020") < 60000), "50-60k")
     .when((f.col("median_total_income_2020") >= 60000) & (f.col("median_total_income_2020") < 70000), "60-70k")
     .when((f.col("median_total_income_2020") >= 70000) & (f.col("median_total_income_2020") < 80000), "70-80k")
     .otherwise("80k+")
)

result_df.show(100, truncate=False)
result_df.printSchema()
result_df.write.csv("../data/curated/merged_postcode_income.csv", header=True, mode="overwrite")